In [41]:
%pip install websockets

Note: you may need to restart the kernel to use updated packages.


In [42]:
import asyncio
import websockets
import json
import time

1) STATE MANAGEMENT ( SUNUCU HAFIZASI )

Bu kısımda aktif öğrenciler için tutacağımız sözlüğü açıyoruz. Anlık durumlar, webscoket kanalı ve şifresi burada tutulacak.

Örnek yapı bu şekilde: {"2300005352": {"ws": <baglanti_kanali>, "state": "in_progress", "session_token": "abc123xyz"}}

In [43]:
active_students = {}

Burada da sınav kayıt defteri var. Sınavların tipleri,süreleri ve bazı özellikleri çeşitli olacağı için onları burada tutacağız ( süreleri,kuralları vs.)

Örnek yapı: {"CS301": {"duration": 60, "prohibited": ["chatgpt"]}}

In [44]:
exam_registry = {}

In [45]:
dashboard_counter = 0  

In [46]:
print("Kütüphaneler yüklendi ve sunucu hafızası oluşturuldu.")

Kütüphaneler yüklendi ve sunucu hafızası oluşturuldu.


2) İLETİŞİM VE KOMUT YÖNETİMİ

Bu kısımda, handle_client fonksiyonu ile, JSON paketleri sunucuya ulaştığında "action" kelimesine göre o anki yapılan işlem ayırt edilerek ilgili işlemler yapılacak.

In [47]:
async def handle_client(websocket):
    global dashboard_counter
    try:
        async for message in websocket:
            data = json.loads(message)
            action = data.get("action")

            if action == "register_exam": # EĞİTMENİN YENİ SINAV KAYIT KOMUTU
                exam_id = data["payload"]["exam_id"]
                exam_registry[exam_id] = data["payload"]
                print(f"\n✅ [EĞİTMEN] Yeni Sınav Aktif Edildi: {exam_id}") 
                await websocket.send(json.dumps({"status": "exam_registered"}))

            elif action == "resume_student": # EĞİTMENİN KOPAN ÖĞRENCİYİ AFFETME VE DEVAM ETTİRME KOMUTU
                hedef_id = data.get("student_id")
                if hedef_id in active_students:
                    active_students[hedef_id]["state"] = "in_progress"
                    print(f"\n🟢 [EĞİTMEN KOMUTU] {hedef_id} numaralı öğrenci affedildi ve DEVAM ETTİRİLDİ.")

            elif action == "request_start_exam": # ÖĞRENCİNİN SINAVA BAŞLAMA VEYA DEVAM ETME TALEBİ
                student_id = data["student_id"]
                exam_id = data["exam_id"]
                

                # AŞAĞIYI ŞİMDİLİK YORUM SATIRI YAPTIM, ÇÜNKÜ SİSTEMİ NASIL YAPACAĞIMIZA KARAR VEREMEDİK. İLERDE EĞİTMENİN KAYITLI SINAVLAR ARASINDAN SEÇMESİ GİBİ BİR ŞEY EKLEMEK İSTİYORUZ.

               # if exam_id not in exam_registry: # Sınav ID'si geçerli mi kontrolü
               #     await websocket.send(json.dumps({"status": "error", "message": f"HATA: '{exam_id}' bulunamadı!"}))
               #     continue
                
                # ÇOKLU GİRİŞ VE FARKLI SINAV KONTROLÜ 
                if student_id in active_students:
                    mevcut_durum = active_students[student_id]["state"]
                    mevcut_sinav = active_students[student_id]["exam_id"]

                    # Kural 1: Öğrenci zaten içeride ve sınavı devam ediyorsa yeni cihazı REDDET!
                    if mevcut_durum == "in_progress":
                        await websocket.send(json.dumps({
                            "status": "error", 
                            "message": f"HATA: Zaten '{mevcut_sinav}' sınavındasınız. Aynı anda iki yerden girilemez!"
                        }))
                        continue
                    
                    # Kural 2: Kopmuş/Dondurulmuş ama FARKLI sınava girmeye çalışıyor -> REDDET!
                    elif mevcut_durum in ["disconnected_paused", "violation_paused"] and mevcut_sinav != exam_id:
                        await websocket.send(json.dumps({
                            "status": "error", 
                            "message": f"HATA: Önce '{mevcut_sinav}' sınavını bitirmelisiniz. '{exam_id}' sınavına geçemezsiniz!"
                        }))
                        continue

                    # Kural 3: Kopmuş/Dondurulmuş ve AYNI sınava girmeye çalışıyor -> KURTAR (Reconnect)
                    elif mevcut_durum in ["disconnected_paused", "violation_paused"] and mevcut_sinav == exam_id:
                        kalan_saniye = active_students[student_id]["time_left"]
                        session_token = active_students[student_id]["session_token"]
                        
                        active_students[student_id]["ws"] = websocket
                        active_students[student_id]["state"] = "in_progress"
                        
                        print(f"\n🔄 [BİLGİ] {student_id} koptuğu yerden tekrar bağlandı!")
                        await websocket.send(json.dumps({
                            "action": "exam_started_ack",
                            "status": "success",
                            "session_token": session_token,
                            "reconnected": True,
                            "time_left_seconds": kalan_saniye
                        }))
                        continue
                # ----------------------------------------------------------------------
                
                # İlk kez giriyorsa normal başlat
                # Yeni Hali (Mert'in DB'si gelene kadar varsayılan 40 dakika veriyoruz):
                duration_mins = 40
                duration_secs = duration_mins * 60
                session_token = f"token_{student_id}_gizli"
                
                active_students[student_id] = {  # öğrencinin durumunu ve bilgilerini tutan sözlük
                    "ws": websocket,
                    "state": "in_progress",
                    "session_token": session_token,
                    "exam_id": exam_id,
                    "time_left": duration_secs
                }
                await websocket.send(json.dumps({ # Öğrenciye sınav başlama onayı ve oturum bilgisi gönder
                    "action": "exam_started_ack",
                    "status": "success",
                    "session_token": session_token,
                    "reconnected": False,
                    "total_duration_minutes": duration_mins 
                }))

            elif action == "status_update":  # ÖĞRENCİNİN SÜREÇ İÇİNDEKİ DURUM GÜNCELLEMESİ
                student_id = data.get("student_id")
                token = data.get("session_token")

                # Naz'ın auth modülüyle şifreli token doğrulaması buraya gelecek
                if student_id in active_students and active_students[student_id]["session_token"] == token:
                    
                    
                    security_data = data.get("security", {})
                    
                    # Ahmet'ten gelen ihlal uyarısı True ise detayları işle
                    if security_data.get("violation_alert") == True:
                        active_students[student_id]["state"] = "violation_paused"
                        
                        # Ahmet ve Engin'in gönderdiği detayları ayıklıyoruz
                        v_type = security_data.get("violation_type", "Bilinmeyen İhlal")
                        details = security_data.get("details", {})
                        aktif_pencere = details.get("active_window", "Bilinmiyor")
                        zaman = security_data.get("timestamp", time.strftime("%H:%M:%S"))
                        
                        # İrem ve Rana'nın Dashboard'da göstermesi için RAM'e kaydediyoruz
                        active_students[student_id]["last_violation"] = {
                            "type": v_type,
                            "window": aktif_pencere,
                            "time": zaman
                        }
                        
                        print(f"\n🚨 [ALARM] {student_id} ihlal yaptı! Sınav donduruldu.")
                        print(f"   ↳ Sebep: {v_type} | Pencere: {aktif_pencere}")
                        
                        # Mert'in DB modülü bittiğinde kayıt fonksiyonu buraya eklenecek
                        # db.save_violation_to_sql(student_id, v_type, aktif_pencere)
 
            elif action == "get_dashboard_data":
                dashboard_counter += 1
                
                # İrem ve Rana için veriyi sunucu tarafında hazırlıyoruz
                formatted_students = {}
                for sid, info in active_students.items():
                    raw_seconds = info.get("time_left", 0)
                    
                    # Saniyeyi "DAKİKA:SANİYE" formatına çeviriyoruz
                    mins = raw_seconds // 60
                    secs = raw_seconds % 60
                    time_str = f"{mins:02d}:{secs:02d}" # Örn: "14:05"
                    
                    formatted_students[sid] = {
                        "state": info["state"], 
                        "exam_id": info.get("exam_id"), 
                        "time_left": raw_seconds,
                        "time_left_formatted": time_str # Yeni eklediğimiz alan
                    }

                dashboard_data = {
                    "action": "dashboard_update",
                    "active_students_count": len(active_students),
                    "students": formatted_students
                }
                await websocket.send(json.dumps(dashboard_data))
                print(f"\r📊 [SİSTEM] Dashboard güncellendi. (İstek: {dashboard_counter})", end="", flush=True)

    except Exception as e:
        # Eski hali: pass  (Hataları gizliyordu)
        print(f"❌ [SİSTEM HATASI] Beklenmedik bir sorun oluştu: {e}")
    finally:
        for sid, info in active_students.items():
            if info["ws"] == websocket:
                info["ws"] = None # YENİ: Kopan öğrencinin bozuk ağ kablosunu RAM'den temizle
                
                # SADECE in_progress ise durumu değiştir
                if info["state"] == "in_progress":
                    if info.get("time_left", 1) <= 0:
                        info["state"] = "completed"
                        print(f"\n✅ [BİTTİ] Öğrenci {sid} süresini doldurdu ve sınavı tamamladı.")
                    else:
                        info["state"] = "disconnected_paused"
                        print(f"\n🔌 [KOPTU] Öğrenci {sid} hattan düştü. Durumu donduruldu.")
                break

print("✅ Sunucu güncellendi!")

✅ Sunucu güncellendi!


3) SUNUCUYU BAŞLATMA VE AÇIK TUTMA

In [49]:
print("🌐 [SUNUCU] Merkezi Sınav Sunucusu Başlatılıyor...")

async def global_timer():
    while True:
        await asyncio.sleep(1) # Her 1 saniyede bir tetiklenir

        for sid, info in active_students.items():
            if info["state"] == "in_progress":
                info["time_left"] -= 1 # Süreyi sunucuda azalt

                if info["time_left"] <= 0:
                    info["state"] = "completed"
                    print(f"\n✅ [BİTTİ] {sid} numaralı öğrencinin SÜRESİ DOLDU!")

async def run_server():
    
    global active_students, exam_registry, dashboard_counter
    active_students.clear()
    exam_registry.clear()
    dashboard_counter = 0
    
    print("🧹 [SİSTEM] Eski oturumlar, sınavlar ve loglar RAM'den tamamen silindi!")
    
    #  EKSİK OLAN SATIR: Sunucu sayacını arka planda başlat
    asyncio.create_task(global_timer())

    async with websockets.serve(handle_client, "0.0.0.0", 8765):
        print("🌐 [SUNUCU] Port 8765 dinleniyor. İstemciler bekleniyor...\n")
        try:
            await asyncio.Future()  
        except asyncio.CancelledError:
            print("🛑 [SİSTEM] Sunucu manuel olarak durduruldu. Kapatılıyor...")

await run_server()

🌐 [SUNUCU] Merkezi Sınav Sunucusu Başlatılıyor...
🧹 [SİSTEM] Eski oturumlar, sınavlar ve loglar RAM'den tamamen silindi!
🌐 [SUNUCU] Port 8765 dinleniyor. İstemciler bekleniyor...

📊 [SİSTEM] Dashboard güncellendi. (İstek: 71)🛑 [SİSTEM] Sunucu manuel olarak durduruldu. Kapatılıyor...
